# Notebook 04 — Insights & Visualisations

**Goal:** Produce all interactive Plotly charts and derive actionable personalization recommendations for each listener segment.

**Research question:** *What user segments exist based on listening diversity patterns, and how can streaming platforms use these insights to optimise personalization strategies for different listener types?*

**Inputs** (from `data/processed/`):
- `user_features.parquet`
- `cluster_labels.parquet`
- `scrobbles_updated.parquet`
- `artist_genres.parquet`
- `profiles.parquet`

**Outputs** (in `outputs/figures/`):
- `umap_clusters.html` — 2D UMAP scatter
- `cluster_heatmap.html` — feature profile heatmap
- `radar_chart.html` — radar comparison of segments
- `feature_importance.html` — discriminating features
- `genre_distribution.html` — genre breakdown per segment
- `temporal_heatmap_cluster_N.html` — listening patterns per segment

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO)

import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = 'notebook'
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
labels_df      = pd.read_parquet('../data/processed/cluster_labels.parquet')
scrobbles      = pd.read_parquet('../data/processed/scrobbles_updated.parquet')
artist_genres  = pd.read_parquet('../data/processed/artist_genres.parquet')
profiles       = pd.read_parquet('../data/processed/profiles.parquet')

# Ensure userid dtype consistency across all DataFrames
for df in [feature_matrix, labels_df, scrobbles, profiles]:
    if 'userid' in df.columns:
        df['userid'] = df['userid'].astype(str).str.strip()

labels        = labels_df['cluster'].values
cluster_names = dict(zip(labels_df['cluster'], labels_df['cluster_name']))
userids       = labels_df['userid'].tolist()

n_segments = labels_df['cluster'].nunique()
print(f'Users: {len(userids)} | Segments: {n_segments}')
print(f'Feature matrix: {feature_matrix.shape}')
print(f'Scrobbles: {len(scrobbles):,} rows')
print()
labels_df['cluster_name'].value_counts()

## 1. UMAP Cluster Scatter

In [ ]:
from src.visualization.plots import plot_umap_clusters, save_figure

umap_coords = labels_df[['umap_x', 'umap_y']].values

fig_umap = plot_umap_clusters(
    umap_coords=umap_coords,
    labels=labels,
    cluster_names=cluster_names,
    userids=userids,
)
save_figure(fig_umap, '../outputs/figures/umap_clusters')
fig_umap.show()

## 2. Cluster Feature Heatmap

Z-scored means — red = above average, blue = below average.

In [ ]:
from src.visualization.plots import plot_cluster_heatmap
from src.clustering.evaluation import summarise_clusters

# Focus on the most interpretable features for the heatmap
key_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists', 'artist_concentration_20',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min', 'weekend_ratio',
    'temporal_hour_entropy', 'morning_ratio', 'evening_ratio',
    'mean_energy', 'mean_valence', 'mean_danceability', 'mean_acousticness',
]
key_features = [f for f in key_features if f in feature_matrix.columns]

fig_heatmap = plot_cluster_heatmap(
    feature_matrix=feature_matrix,
    labels=labels,
    features=key_features,
    cluster_names=cluster_names,
)
save_figure(fig_heatmap, '../outputs/figures/cluster_heatmap')
fig_heatmap.show()

## 3. Radar Chart — Segment Profiles

In [ ]:
from src.visualization.plots import plot_cluster_radar

cluster_summary = summarise_clusters(feature_matrix, labels)

radar_features = [
    'artist_entropy', 'genre_entropy', 'novelty_ratio',
    'track_replay_rate', 'avg_tracks_per_session', 'temporal_hour_entropy',
]
radar_features = [f for f in radar_features if f in feature_matrix.columns]

fig_radar = plot_cluster_radar(
    cluster_summary=cluster_summary,
    features=radar_features,
    cluster_names=cluster_names,
)
save_figure(fig_radar, '../outputs/figures/radar_chart')
fig_radar.show()

## 4. Feature Importance

In [ ]:
from src.clustering.evaluation import feature_importance
from src.visualization.plots import plot_feature_importance

imp_df = feature_importance(feature_matrix, labels)

fig_imp = plot_feature_importance(imp_df, top_n=20)
save_figure(fig_imp, '../outputs/figures/feature_importance')
fig_imp.show()

## 5. Genre Distribution by Segment

In [ ]:
from src.visualization.plots import plot_genre_distribution

fig_genre = plot_genre_distribution(
    scrobbles=scrobbles,
    artist_genres=artist_genres,
    labels=labels,
    userids=userids,
    top_n_genres=12,
    cluster_names=cluster_names,
)
save_figure(fig_genre, '../outputs/figures/genre_distribution')
fig_genre.show()

## 6. Temporal Listening Patterns (per segment)

In [ ]:
from src.visualization.plots import plot_temporal_heatmap

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    fig_temp = plot_temporal_heatmap(
        scrobbles=scrobbles,
        labels=labels,
        userids=userids,
        cluster_id=cid,
        cluster_name=cluster_names.get(cid, f'Cluster {cid}'),
    )
    save_figure(fig_temp, f'../outputs/figures/temporal_heatmap_cluster_{cid}')
    fig_temp.show()

## 7. Business Insights & Personalization Recommendations

The cell below prints a structured summary of each segment's characteristics and actionable implications for streaming platforms.

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters

cluster_summary = summarise_clusters(feature_matrix, labels)
cluster_names_auto = label_clusters(cluster_summary, feature_matrix, labels)

# Key feature means per cluster (unscaled)
fm = feature_matrix.copy()
fm['cluster'] = labels

insight_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'weekend_ratio', 'temporal_hour_entropy',
]
insight_features = [f for f in insight_features if f in fm.columns]

global_means = fm[insight_features].mean()

print('=' * 72)
print('LISTENER SEGMENT PROFILES & PERSONALIZATION RECOMMENDATIONS')
print('=' * 72)

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    cluster_users = fm[fm['cluster'] == cid]
    n = len(cluster_users)
    name = cluster_names.get(cid, f'Cluster {cid}')
    means = cluster_users[insight_features].mean()

    print(f'\nSEGMENT {cid}: {name.upper()}  ({n} users, {n/len(fm)*100:.1f}%)')
    print('-' * 60)

    for feat in insight_features:
        val = means[feat]
        gval = global_means[feat]
        delta = (val - gval) / (gval + 1e-9)
        arrow = '▲' if delta > 0.15 else ('▼' if delta < -0.15 else '—')
        print(f'  {feat:<35} {val:8.3f}  {arrow} (global: {gval:.3f})')

    # Heuristic recommendations
    recs = []
    if means.get('novelty_ratio', 0) > global_means.get('novelty_ratio', 0):
        recs.append('→ Prioritise new artist recommendations and discovery playlists')
    else:
        recs.append('→ Emphasise "More like your favourites" and artist radio')

    if means.get('track_replay_rate', 0) > global_means.get('track_replay_rate', 0) * 1.2:
        recs.append('→ Offer offline mode / download prompts for favourite tracks')

    if means.get('genre_entropy', 0) > global_means.get('genre_entropy', 0):
        recs.append('→ Cross-genre mood playlists and genre-blend features work well')
    else:
        recs.append('→ Deep-dive genre playlists and artist discography features')

    if means.get('weekend_ratio', 0) > 0.45:
        recs.append('→ Target weekend push notifications and curated weekend playlists')

    if means.get('avg_tracks_per_session', 0) > global_means.get('avg_tracks_per_session', 0) * 1.3:
        recs.append('→ Long-session features: auto-queuing, seamless transitions, sleep timer')

    print('\n  PLATFORM RECOMMENDATIONS:')
    for r in recs:
        print(f'  {r}')

print('\n' + '=' * 72)

## 8. Demographic Breakdown By Listener Segment

Visualise how the discovered clusters split across gender, age, and country.
Each chart is saved as an individual EPS file for publication use.
Users with no demographic data recorded are excluded from the respective chart
and the excluded count is annotated on the figure.

In [ ]:
from src.visualization.plots import plot_demographic_breakdown
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# Build the joined table once for use in all demographic charts
demo_cols = profiles[['userid', 'gender', 'age', 'country']].copy()
demo_cols['userid'] = demo_cols['userid'].astype(str).str.strip()
labeled = labels_df.merge(demo_cols, on='userid', how='left')

# ── Gender Distribution Per Segment ──────────────────────────────────────────
gender_data = labeled.dropna(subset=['gender'])
gender_data = gender_data[gender_data['gender'].str.strip() != '']

if not gender_data.empty:
    gender_ct  = gender_data.groupby(['cluster_name', 'gender']).size().unstack(fill_value=0)
    gender_pct = gender_ct.div(gender_ct.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#636EFA', '#EF553B', '#00CC96']
    gender_pct.plot(kind='bar', ax=ax, color=colors[:len(gender_pct.columns)],
                    edgecolor='white', width=0.7)
    ax.set_title('Gender Distribution By Listener Segment', fontsize=13, fontweight='bold')
    ax.set_xlabel('Listener Segment')
    ax.set_ylabel('Percentage Of Users (%)')
    ax.legend(title='Gender', bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=30)
    n_na = len(labeled) - len(gender_data)
    ax.annotate(f'Note: {n_na} users with no gender recorded are excluded.',
                xy=(0.01, 0.01), xycoords='axes fraction', fontsize=8, color='grey')
    plt.tight_layout()
    plt.savefig(figures_dir / 'segment_gender_distribution.eps', format='eps', bbox_inches='tight')
    plt.savefig(figures_dir / 'segment_gender_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gender distribution per segment (%):')
    print(gender_pct.round(1).to_string())
else:
    print('No gender data available.')

# ── Median Age Per Segment ────────────────────────────────────────────────────
age_data = labeled.dropna(subset=['age'])
age_data = age_data[(age_data['age'] >= 10) & (age_data['age'] <= 100)]

if not age_data.empty:
    median_age = age_data.groupby('cluster_name')['age'].median().sort_values()

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(median_age.index, median_age.values, color='#636EFA', edgecolor='white')
    ax.set_title('Median Age Per Listener Segment', fontsize=13, fontweight='bold')
    ax.set_xlabel('Median Age (Years)')
    ax.set_ylabel('Listener Segment')
    for bar, val in zip(bars, median_age.values):
        ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(figures_dir / 'segment_median_age.eps', format='eps', bbox_inches='tight')
    plt.savefig(figures_dir / 'segment_median_age.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Age histogram overlaid per segment
    palette = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (seg, grp) in enumerate(age_data.groupby('cluster_name')):
        grp['age'].hist(ax=ax, bins=15, alpha=0.55, label=seg,
                        color=palette[i % len(palette)], edgecolor='white')
    ax.set_title('Age Distribution By Listener Segment', fontsize=13, fontweight='bold')
    ax.set_xlabel('Age (Years)')
    ax.set_ylabel('Number Of Users')
    ax.legend(title='Segment', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(figures_dir / 'segment_age_distribution.eps', format='eps', bbox_inches='tight')
    plt.savefig(figures_dir / 'segment_age_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nMedian age per segment:')
    print(median_age.round(1).to_string())
else:
    print('No age data available.')

# ── Top Countries Per Segment ─────────────────────────────────────────────────
country_data = labeled.dropna(subset=['country'])
country_data = country_data[country_data['country'].str.strip() != '']

if not country_data.empty:
    top_countries = country_data['country'].value_counts().head(6).index.tolist()
    c_seg = country_data[country_data['country'].isin(top_countries)]

    if not c_seg.empty:
        country_ct  = c_seg.groupby(['cluster_name', 'country']).size().unstack(fill_value=0)
        country_pct = country_ct.div(country_ct.sum(axis=1), axis=0) * 100

        fig, ax = plt.subplots(figsize=(12, 5))
        country_pct.plot(kind='bar', ax=ax, edgecolor='white', width=0.7)
        ax.set_title('Top Country Distribution By Listener Segment',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Listener Segment')
        ax.set_ylabel('Percentage Of Users (%)')
        ax.legend(title='Country', bbox_to_anchor=(1.01, 1), loc='upper left')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.savefig(figures_dir / 'segment_country_distribution.eps', format='eps', bbox_inches='tight')
        plt.savefig(figures_dir / 'segment_country_distribution.png', dpi=150, bbox_inches='tight')
        plt.show()

    print('\nTop 3 countries per segment:')
    for seg in sorted(country_data['cluster_name'].unique()):
        top3 = (country_data[country_data['cluster_name'] == seg]['country']
                .value_counts().head(3).index.tolist())
        print(f'  {seg}: {", ".join(top3)}')
else:
    print('No country data available.')